# Pipeline de Dados — Absenteísmo

Este notebook apresenta a construção do pipeline de dados do projeto de análise de absenteísmo, desenvolvido no Databricks utilizando a arquitetura Medallion.

O processamento está organizado nas camadas **Bronze, Silver e Gold**, contemplando a ingestão dos dados brutos, avaliações de qualidade, padronização e enriquecimento dos atributos e, por fim, a criação de tabelas agregadas destinadas às análises de negócio.

In [0]:
%sql
-- ============================================
-- 1. CONFIGURAÇÃO DO AMBIENTE
-- ============================================

SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
-- Criação dos schemas da arquitetura Medallion

CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
-- Validação dos schemas criados

SHOW SCHEMAS IN workspace;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%python
# ============================================
# 2. INGESTÃO DOS DADOS
# ============================================

# Caminho do arquivo-fonte armazenado no Volume
caminho_csv = "/Volumes/workspace/bronze/source_files/Absenteeism_at_work.csv"

# Leitura inicial do arquivo CSV
df_absenteismo = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .csv(caminho_csv)
)

display(df_absenteismo)

In [0]:
%python
# Validação da estrutura do arquivo

print(f"Quantidade de registros: {df_absenteismo.count()}")
print(f"Quantidade de colunas: {len(df_absenteismo.columns)}")

df_absenteismo.printSchema()

Quantidade de registros: 740
Quantidade de colunas: 21
root
 |-- ID: string (nullable = true)
 |-- Reason for absence: string (nullable = true)
 |-- Month of absence: string (nullable = true)
 |-- Day of the week: string (nullable = true)
 |-- Seasons: string (nullable = true)
 |-- Transportation expense: string (nullable = true)
 |-- Distance from Residence to Work: string (nullable = true)
 |-- Service time: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Work load Average/day : string (nullable = true)
 |-- Hit target: string (nullable = true)
 |-- Disciplinary failure: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Son: string (nullable = true)
 |-- Social drinker: string (nullable = true)
 |-- Social smoker: string (nullable = true)
 |-- Pet: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- Height: string (nullable = true)
 |-- Body mass index: string (nullable = true)
 |-- Absenteeism time in hours: string (nullable = tr

In [0]:
%python
# ============================================
# 3. CAMADA BRONZE
# ============================================

# Gravação dos dados brutos no formato Delta,
# preservando os nomes originais das colunas

(
    df_absenteismo.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .saveAsTable("workspace.bronze.absenteeism_raw")
)

print("Tabela Bronze criada com sucesso.")

Tabela Bronze criada com sucesso.


In [0]:
-- Validação da tabela Bronze

SELECT *
FROM workspace.bronze.absenteeism_raw
LIMIT 10;

ID,Reason for absence,Month of absence,Day of the week,Seasons,Transportation expense,Distance from Residence to Work,Service time,Age,Work load Average/day,Hit target,Disciplinary failure,Education,Son,Social drinker,Social smoker,Pet,Weight,Height,Body mass index,Absenteeism time in hours
11,26,7,3,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,4
36,0,7,3,1,118,13,18,50,239.554,97,1,1,1,1,0,0,98,178,31,0
3,23,7,4,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2
7,7,7,5,1,279,5,14,39,239.554,97,0,1,2,1,1,0,68,168,24,4
11,23,7,5,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,2
3,23,7,6,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2
10,22,7,6,1,361,52,3,28,239.554,97,0,1,1,1,0,4,80,172,27,8
20,23,7,6,1,260,50,11,36,239.554,97,0,1,4,1,0,0,65,168,23,4
14,19,7,2,1,155,12,14,34,239.554,97,0,1,2,1,0,0,95,196,25,40
1,22,7,2,1,235,11,14,37,239.554,97,0,3,1,0,0,1,88,172,29,8


In [0]:
-- Validação da quantidade de registros da Bronze

SELECT COUNT(*) AS quantidade_registros
FROM workspace.bronze.absenteeism_raw;

quantidade_registros
740


In [0]:
%python
# ============================================
# 4. QUALIDADE DOS DADOS
# ============================================

# Carregamento da tabela Bronze para análise de qualidade
df_bronze = spark.table("workspace.bronze.absenteeism_raw")

print(f"Quantidade de registros: {df_bronze.count()}")
print(f"Quantidade de colunas: {len(df_bronze.columns)}")
print(f"Quantidade de colaboradores distintos: {df_bronze.select('ID').distinct().count()}")

Quantidade de registros: 740
Quantidade de colunas: 21
Quantidade de colaboradores distintos: 36


In [0]:
%python

# --------------------------------------------
# 4.1 ANÁLISE DE COMPLETUDE
# --------------------------------------------

# Importação das funções utilizadas nas análises de qualidade
from pyspark.sql import functions as F

# Verificação de valores nulos ou vazios por atributo
total_registros = df_bronze.count()
completude = []

for coluna in df_bronze.columns:
    qtd_ausentes = df_bronze.filter(
        F.col(coluna).isNull() | (F.trim(F.col(coluna)) == "")
    ).count()

    completude.append(
        (
            coluna,
            total_registros,
            qtd_ausentes,
            round(((total_registros - qtd_ausentes) / total_registros) * 100, 2)
        )
    )

df_relatorio_completude = spark.createDataFrame(
    completude,
    [
        "atributo",
        "total_registros",
        "valores_ausentes",
        "completude_percentual"
    ]
)

display(df_relatorio_completude)

atributo,total_registros,valores_ausentes,completude_percentual
ID,740,0,100.0
Reason for absence,740,0,100.0
Month of absence,740,0,100.0
Day of the week,740,0,100.0
Seasons,740,0,100.0
Transportation expense,740,0,100.0
Distance from Residence to Work,740,0,100.0
Service time,740,0,100.0
Age,740,0,100.0
Work load Average/day,740,0,100.0


In [0]:
%python

# --------------------------------------------
# 4.2 Análise de unicidade e duplicidade
# --------------------------------------------

total_registros = df_bronze.count()
registros_distintos = df_bronze.distinct().count()
registros_duplicados = total_registros - registros_distintos

print(f"Total de registros: {total_registros}")
print(f"Registros distintos: {registros_distintos}")
print(f"Registros duplicados: {registros_duplicados}")

Total de registros: 740
Registros distintos: 706
Registros duplicados: 34


In [0]:
%python
# Identificação dos grupos de registros duplicados

df_duplicados = (
    df_bronze
    .groupBy(*df_bronze.columns)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

display(df_duplicados)

ID,Reason for absence,Month of absence,Day of the week,Seasons,Transportation expense,Distance from Residence to Work,Service time,Age,Work load Average/day,Hit target,Disciplinary failure,Education,Son,Social drinker,Social smoker,Pet,Weight,Height,Body mass index,Absenteeism time in hours,count
3,27,2,4,2,179,51,18,38,251.818,96,0,1,0,1,0,0,89,170,31,3,4
22,27,4,6,3,179,26,9,30,246.288,91,0,3,0,0,0,0,56,171,19,2,4
3,27,3,4,2,179,51,18,38,222.196,99,0,1,0,1,0,0,89,170,31,2,3
3,27,2,4,2,179,51,18,38,264.249,97,0,1,0,1,0,0,89,170,31,2,3
3,27,2,6,2,179,51,18,38,251.818,96,0,1,0,1,0,0,89,170,31,3,3
3,27,3,5,2,179,51,18,38,222.196,99,0,1,0,1,0,0,89,170,31,3,3
3,28,12,6,4,179,51,18,38,236.629,93,0,1,0,1,0,0,89,170,31,1,2
3,27,2,2,2,179,51,18,38,264.249,97,0,1,0,1,0,0,89,170,31,2,2
15,28,5,5,3,291,31,12,40,237.656,99,0,1,1,1,0,1,73,171,25,2,2
34,27,1,4,2,118,10,10,37,308.593,95,0,1,0,0,0,0,83,172,28,2,2


In [0]:
%python
# --------------------------------------------
# 4.3 ANÁLISE DE CONSISTÊNCIA
# --------------------------------------------

# Domínios esperados conforme documentação da fonte
dominios = {
    "Reason for absence": list(range(1, 29)),
    "Month of absence": list(range(1, 13)),
    "Day of the week": [2, 3, 4, 5, 6],
    "Seasons": [1, 2, 3, 4],
    "Disciplinary failure": [0, 1],
    "Education": [1, 2, 3, 4],
    "Social drinker": [0, 1],
    "Social smoker": [0, 1]
}

resultado_consistencia = []

for coluna, dominio in dominios.items():

    # Conversão temporária para inteiro apenas para a validação
    coluna_int = F.col(coluna).cast("int")

    qtd_inconsistentes = (
        df_bronze
        .filter(
            coluna_int.isNull() |
            (~coluna_int.isin(dominio))
        )
        .count()
    )

    resultado_consistencia.append(
        (
            coluna,
            str(dominio),
            qtd_inconsistentes
        )
    )

df_consistencia = spark.createDataFrame(
    resultado_consistencia,
    [
        "atributo",
        "dominio_esperado",
        "valores_fora_dominio"
    ]
)

display(df_consistencia)

atributo,dominio_esperado,valores_fora_dominio
Reason for absence,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]",43
Month of absence,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",3
Day of the week,"[2, 3, 4, 5, 6]",0
Seasons,"[1, 2, 3, 4]",0
Disciplinary failure,"[0, 1]",0
Education,"[1, 2, 3, 4]",0
Social drinker,"[0, 1]",0
Social smoker,"[0, 1]",0


In [0]:
%python
# Detalhamento dos valores encontrados fora dos domínios esperados

campos_inconsistentes = {
    "Reason for absence": list(range(1, 29)),
    "Month of absence": list(range(1, 13))
}

for coluna, dominio in campos_inconsistentes.items():

    print(f"\nValores fora do domínio - {coluna}")

    (
        df_bronze
        .filter(~F.col(coluna).cast("int").isin(dominio))
        .groupBy(coluna)
        .count()
        .orderBy(coluna)
        .show()
    )


Valores fora do domínio - Reason for absence
+------------------+-----+
|Reason for absence|count|
+------------------+-----+
|                 0|   43|
+------------------+-----+


Valores fora do domínio - Month of absence
+----------------+-----+
|Month of absence|count|
+----------------+-----+
|               0|    3|
+----------------+-----+



In [0]:
%python
# --------------------------------------------
# 4.4 ANÁLISE DE PLAUSIBILIDADE
# --------------------------------------------

# Atributos numéricos avaliados quanto à plausibilidade
colunas_numericas = [
    "Transportation expense",
    "Distance from Residence to Work",
    "Service time",
    "Age",
    "Work load Average/day ",
    "Hit target",
    "Son",
    "Pet",
    "Weight",
    "Height",
    "Body mass index",
    "Absenteeism time in hours"
]

resultado_plausibilidade = []

for coluna in colunas_numericas:
    
    valores = df_bronze.select(
        F.col(coluna).cast("double").alias("valor")
    )
    
    estatisticas = valores.agg(
        F.min("valor").alias("minimo"),
        F.max("valor").alias("maximo"),
        F.sum(
            F.when(F.col("valor") < 0, 1).otherwise(0)
        ).alias("valores_negativos")
    ).first()
    
    resultado_plausibilidade.append(
        (
            coluna.strip(),
            estatisticas["minimo"],
            estatisticas["maximo"],
            estatisticas["valores_negativos"]
        )
    )

df_plausibilidade = spark.createDataFrame(
    resultado_plausibilidade,
    [
        "atributo",
        "valor_minimo",
        "valor_maximo",
        "valores_negativos"
    ]
)

display(df_plausibilidade)

atributo,valor_minimo,valor_maximo,valores_negativos
Transportation expense,118.0,388.0,0
Distance from Residence to Work,5.0,52.0,0
Service time,1.0,29.0,0
Age,27.0,58.0,0
Work load Average/day,205.917,378.884,0
Hit target,81.0,100.0,0
Son,0.0,4.0,0
Pet,0.0,8.0,0
Weight,56.0,108.0,0
Height,163.0,196.0,0


In [0]:
%python
# Verificação de registros com ausência igual a zero hora

qtd_ausencia_zero = (
    df_bronze
    .filter(F.col("Absenteeism time in hours").cast("double") == 0)
    .count()
)

print(f"Registros com 0 hora de absenteísmo: {qtd_ausencia_zero}")

Registros com 0 hora de absenteísmo: 44


In [0]:
%python
# --------------------------------------------
# 4.5 ANÁLISE DE OUTLIERS
# --------------------------------------------

coluna_horas = "Absenteeism time in hours"

# Conversão temporária para análise estatística
df_horas = df_bronze.select(
    F.col(coluna_horas).cast("double").alias("horas")
)

# Cálculo dos quartis
q1, q3 = df_horas.approxQuantile(
    "horas",
    [0.25, 0.75],
    0.0
)

iqr = q3 - q1

limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

# Identificação dos outliers
qtd_outliers = (
    df_horas
    .filter(
        (F.col("horas") < limite_inferior) |
        (F.col("horas") > limite_superior)
    )
    .count()
)

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Limite inferior: {limite_inferior}")
print(f"Limite superior: {limite_superior}")
print(f"Quantidade de outliers: {qtd_outliers}")

Q1: 2.0
Q3: 8.0
IQR: 6.0
Limite inferior: -7.0
Limite superior: 17.0
Quantidade de outliers: 44


In [0]:
%python
# Detalhamento dos valores classificados como outliers

df_outliers = (
    df_horas
    .filter(
        (F.col("horas") < limite_inferior) |
        (F.col("horas") > limite_superior)
    )
    .groupBy("horas")
    .count()
    .orderBy("horas")
)

display(df_outliers)

horas,count
24.0,16
32.0,6
40.0,7
48.0,1
56.0,2
64.0,3
80.0,3
104.0,1
112.0,2
120.0,3


In [0]:
%python
# ============================================
# 5. CAMADA SILVER
# ============================================

# --------------------------------------------
# 5.1 PADRONIZAÇÃO E TIPAGEM DOS DADOS
# --------------------------------------------

df_silver = df_bronze.select(
    F.col("ID").cast("int").alias("id"),
    F.col("Reason for absence").cast("int").alias("reason_code"),
    F.col("Month of absence").cast("int").alias("month_code"),
    F.col("Day of the week").cast("int").alias("day_of_week_code"),
    F.col("Seasons").cast("int").alias("season_code"),
    F.col("Transportation expense").cast("int").alias("transportation_expense"),
    F.col("Distance from Residence to Work").cast("int").alias("distance_from_residence_to_work"),
    F.col("Service time").cast("int").alias("service_time"),
    F.col("Age").cast("int").alias("age"),
    F.col("Work load Average/day ").cast("double").alias("work_load_average_day"),
    F.col("Hit target").cast("int").alias("hit_target"),
    F.col("Disciplinary failure").cast("int").alias("disciplinary_failure"),
    F.col("Education").cast("int").alias("education_code"),
    F.col("Son").cast("int").alias("son"),
    F.col("Social drinker").cast("int").alias("social_drinker"),
    F.col("Social smoker").cast("int").alias("social_smoker"),
    F.col("Pet").cast("int").alias("pet"),
    F.col("Weight").cast("int").alias("weight"),
    F.col("Height").cast("int").alias("height"),
    F.col("Body mass index").cast("int").alias("body_mass_index"),
    F.col("Absenteeism time in hours").cast("int").alias("absenteeism_time_in_hours")
)

display(df_silver)

id,reason_code,month_code,day_of_week_code,season_code,transportation_expense,distance_from_residence_to_work,service_time,age,work_load_average_day,hit_target,disciplinary_failure,education_code,son,social_drinker,social_smoker,pet,weight,height,body_mass_index,absenteeism_time_in_hours
11,26,7,3,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,4
36,0,7,3,1,118,13,18,50,239.554,97,1,1,1,1,0,0,98,178,31,0
3,23,7,4,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2
7,7,7,5,1,279,5,14,39,239.554,97,0,1,2,1,1,0,68,168,24,4
11,23,7,5,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,2
3,23,7,6,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2
10,22,7,6,1,361,52,3,28,239.554,97,0,1,1,1,0,4,80,172,27,8
20,23,7,6,1,260,50,11,36,239.554,97,0,1,4,1,0,0,65,168,23,4
14,19,7,2,1,155,12,14,34,239.554,97,0,1,2,1,0,0,95,196,25,40
1,22,7,2,1,235,11,14,37,239.554,97,0,3,1,0,0,1,88,172,29,8


In [0]:
%python
# Validação da estrutura após padronização e tipagem

print(f"Quantidade de registros: {df_silver.count()}")
print(f"Quantidade de colunas: {len(df_silver.columns)}")

df_silver.printSchema()

Quantidade de registros: 740
Quantidade de colunas: 21
root
 |-- id: integer (nullable = true)
 |-- reason_code: integer (nullable = true)
 |-- month_code: integer (nullable = true)
 |-- day_of_week_code: integer (nullable = true)
 |-- season_code: integer (nullable = true)
 |-- transportation_expense: integer (nullable = true)
 |-- distance_from_residence_to_work: integer (nullable = true)
 |-- service_time: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- work_load_average_day: double (nullable = true)
 |-- hit_target: integer (nullable = true)
 |-- disciplinary_failure: integer (nullable = true)
 |-- education_code: integer (nullable = true)
 |-- son: integer (nullable = true)
 |-- social_drinker: integer (nullable = true)
 |-- social_smoker: integer (nullable = true)
 |-- pet: integer (nullable = true)
 |-- weight: integer (nullable = true)
 |-- height: integer (nullable = true)
 |-- body_mass_index: integer (nullable = true)
 |-- absenteeism_time_in_hours: intege

In [0]:
%python
# --------------------------------------------
# 5.2 ENRIQUECIMENTO DOS ATRIBUTOS CATEGÓRICOS
# --------------------------------------------

df_silver = (
    df_silver

    # Descrição do mês
    .withColumn(
        "month_name",
        F.when(F.col("month_code") == 1, "Janeiro")
         .when(F.col("month_code") == 2, "Fevereiro")
         .when(F.col("month_code") == 3, "Março")
         .when(F.col("month_code") == 4, "Abril")
         .when(F.col("month_code") == 5, "Maio")
         .when(F.col("month_code") == 6, "Junho")
         .when(F.col("month_code") == 7, "Julho")
         .when(F.col("month_code") == 8, "Agosto")
         .when(F.col("month_code") == 9, "Setembro")
         .when(F.col("month_code") == 10, "Outubro")
         .when(F.col("month_code") == 11, "Novembro")
         .when(F.col("month_code") == 12, "Dezembro")
         .otherwise("Não informado")
    )

    # Descrição do dia da semana
    .withColumn(
        "day_of_week_name",
        F.when(F.col("day_of_week_code") == 2, "Segunda-feira")
         .when(F.col("day_of_week_code") == 3, "Terça-feira")
         .when(F.col("day_of_week_code") == 4, "Quarta-feira")
         .when(F.col("day_of_week_code") == 5, "Quinta-feira")
         .when(F.col("day_of_week_code") == 6, "Sexta-feira")
         .otherwise("Não informado")
    )

    # Descrição da estação
    .withColumn(
        "season_name",
        F.when(F.col("season_code") == 1, "Verão")
         .when(F.col("season_code") == 2, "Outono")
         .when(F.col("season_code") == 3, "Inverno")
         .when(F.col("season_code") == 4, "Primavera")
         .otherwise("Não informado")
    )

    # Descrição da escolaridade
    .withColumn(
        "education_description",
        F.when(F.col("education_code") == 1, "Ensino médio")
         .when(F.col("education_code") == 2, "Graduação")
         .when(F.col("education_code") == 3, "Pós-graduação")
         .when(F.col("education_code") == 4, "Mestrado/Doutorado")
         .otherwise("Não informado")
    )
)

display(
    df_silver.select(
        "month_code",
        "month_name",
        "day_of_week_code",
        "day_of_week_name",
        "season_code",
        "season_name",
        "education_code",
        "education_description"
    ).distinct()
)

month_code,month_name,day_of_week_code,day_of_week_name,season_code,season_name,education_code,education_description
7,Julho,3,Terça-feira,1,Verão,1,Ensino médio
7,Julho,4,Quarta-feira,1,Verão,1,Ensino médio
7,Julho,5,Quinta-feira,1,Verão,1,Ensino médio
7,Julho,6,Sexta-feira,1,Verão,1,Ensino médio
7,Julho,2,Segunda-feira,1,Verão,1,Ensino médio
7,Julho,2,Segunda-feira,1,Verão,3,Pós-graduação
8,Agosto,4,Quarta-feira,1,Verão,1,Ensino médio
8,Agosto,4,Quarta-feira,1,Verão,2,Graduação
8,Agosto,2,Segunda-feira,1,Verão,1,Ensino médio
8,Agosto,6,Sexta-feira,1,Verão,1,Ensino médio


In [0]:
%python
# Tabela auxiliar com a descrição dos motivos de ausência
# baseada na documentação original do dataset

reason_data = [
    (1, "Doenças infecciosas e parasitárias"),
    (2, "Neoplasias"),
    (3, "Doenças do sangue e transtornos imunológicos"),
    (4, "Doenças endócrinas, nutricionais e metabólicas"),
    (5, "Transtornos mentais e comportamentais"),
    (6, "Doenças do sistema nervoso"),
    (7, "Doenças dos olhos e anexos"),
    (8, "Doenças do ouvido e processo mastoide"),
    (9, "Doenças do sistema circulatório"),
    (10, "Doenças do sistema respiratório"),
    (11, "Doenças do sistema digestivo"),
    (12, "Doenças da pele e tecido subcutâneo"),
    (13, "Doenças do sistema musculoesquelético e tecido conjuntivo"),
    (14, "Doenças do sistema geniturinário"),
    (15, "Gravidez, parto e puerpério"),
    (16, "Condições originadas no período perinatal"),
    (17, "Malformações congênitas e anomalias cromossômicas"),
    (18, "Sintomas, sinais e achados clínicos e laboratoriais"),
    (19, "Lesões, envenenamentos e consequências de causas externas"),
    (20, "Causas externas de morbidade e mortalidade"),
    (21, "Fatores que influenciam o estado de saúde"),
    (22, "Acompanhamento do paciente"),
    (23, "Consulta médica"),
    (24, "Doação de sangue"),
    (25, "Exame laboratorial"),
    (26, "Ausência injustificada"),
    (27, "Fisioterapia"),
    (28, "Consulta odontológica")
]

df_reason_lookup = spark.createDataFrame(
    reason_data,
    ["reason_code", "reason_description"]
)

display(df_reason_lookup)

reason_code,reason_description
1,Doenças infecciosas e parasitárias
2,Neoplasias
3,Doenças do sangue e transtornos imunológicos
4,"Doenças endócrinas, nutricionais e metabólicas"
5,Transtornos mentais e comportamentais
6,Doenças do sistema nervoso
7,Doenças dos olhos e anexos
8,Doenças do ouvido e processo mastoide
9,Doenças do sistema circulatório
10,Doenças do sistema respiratório


In [0]:
%python
# --------------------------------------------
# 5.3 INTEGRAÇÃO DOS MOTIVOS DE AUSÊNCIA
# --------------------------------------------

df_silver = (
    df_silver
    .join(
        df_reason_lookup,
        on="reason_code",
        how="left"
    )
    .withColumn(
        "reason_description",
        F.coalesce(
            F.col("reason_description"),
            F.lit("Não especificado na documentação")
        )
    )
)

display(
    df_silver.select(
        "reason_code",
        "reason_description"
    )
    .groupBy(
        "reason_code",
        "reason_description"
    )
    .count()
    .orderBy("reason_code")
)

reason_code,reason_description,count
0,Não especificado na documentação,43
1,Doenças infecciosas e parasitárias,16
2,Neoplasias,1
3,Doenças do sangue e transtornos imunológicos,1
4,"Doenças endócrinas, nutricionais e metabólicas",2
5,Transtornos mentais e comportamentais,3
6,Doenças do sistema nervoso,8
7,Doenças dos olhos e anexos,15
8,Doenças do ouvido e processo mastoide,6
9,Doenças do sistema circulatório,4


In [0]:
%python
# --------------------------------------------
# 5.4 VALIDAÇÃO DA CAMADA SILVER
# --------------------------------------------

print(f"Registros na Bronze: {df_bronze.count()}")
print(f"Registros na Silver: {df_silver.count()}")
print(f"Quantidade de colunas na Silver: {len(df_silver.columns)}")

# Verificação de valores nulos na descrição dos motivos
qtd_reason_null = (
    df_silver
    .filter(F.col("reason_description").isNull())
    .count()
)

print(f"Descrições de motivo nulas: {qtd_reason_null}")

Registros na Bronze: 740
Registros na Silver: 740
Quantidade de colunas na Silver: 26
Descrições de motivo nulas: 0


In [0]:
%python
# --------------------------------------------
# 5.5 PERSISTÊNCIA DA CAMADA SILVER
# --------------------------------------------

# Gravação da tabela principal tratada
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.absenteeism")
)

# Gravação da tabela auxiliar de motivos
(
    df_reason_lookup.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.reason_lookup")
)

print("Tabelas da camada Silver gravadas com sucesso.")

Tabelas da camada Silver gravadas com sucesso.


In [0]:
%python
# Validação das tabelas persistidas na Silver

df_silver_validacao = spark.table(
    "workspace.silver.absenteeism"
)

df_reason_validacao = spark.table(
    "workspace.silver.reason_lookup"
)

print(
    f"workspace.silver.absenteeism: "
    f"{df_silver_validacao.count()} registros"
)

print(
    f"workspace.silver.reason_lookup: "
    f"{df_reason_validacao.count()} registros"
)

display(df_silver_validacao.limit(10))

workspace.silver.absenteeism: 740 registros
workspace.silver.reason_lookup: 28 registros


reason_code,id,month_code,day_of_week_code,season_code,transportation_expense,distance_from_residence_to_work,service_time,age,work_load_average_day,hit_target,disciplinary_failure,education_code,son,social_drinker,social_smoker,pet,weight,height,body_mass_index,absenteeism_time_in_hours,month_name,day_of_week_name,season_name,education_description,reason_description
26,11,7,3,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,4,Julho,Terça-feira,Verão,Ensino médio,Ausência injustificada
0,36,7,3,1,118,13,18,50,239.554,97,1,1,1,1,0,0,98,178,31,0,Julho,Terça-feira,Verão,Ensino médio,Não especificado na documentação
23,3,7,4,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2,Julho,Quarta-feira,Verão,Ensino médio,Consulta médica
7,7,7,5,1,279,5,14,39,239.554,97,0,1,2,1,1,0,68,168,24,4,Julho,Quinta-feira,Verão,Ensino médio,Doenças dos olhos e anexos
23,11,7,5,1,289,36,13,33,239.554,97,0,1,2,1,0,1,90,172,30,2,Julho,Quinta-feira,Verão,Ensino médio,Consulta médica
23,3,7,6,1,179,51,18,38,239.554,97,0,1,0,1,0,0,89,170,31,2,Julho,Sexta-feira,Verão,Ensino médio,Consulta médica
22,10,7,6,1,361,52,3,28,239.554,97,0,1,1,1,0,4,80,172,27,8,Julho,Sexta-feira,Verão,Ensino médio,Acompanhamento do paciente
23,20,7,6,1,260,50,11,36,239.554,97,0,1,4,1,0,0,65,168,23,4,Julho,Sexta-feira,Verão,Ensino médio,Consulta médica
19,14,7,2,1,155,12,14,34,239.554,97,0,1,2,1,0,0,95,196,25,40,Julho,Segunda-feira,Verão,Ensino médio,"Lesões, envenenamentos e consequências de causas externas"
22,1,7,2,1,235,11,14,37,239.554,97,0,3,1,0,0,1,88,172,29,8,Julho,Segunda-feira,Verão,Pós-graduação,Acompanhamento do paciente


In [0]:
%python
# ============================================
# 6. CAMADA GOLD
# ============================================

# Leitura da tabela Silver persistida
df_silver_gold = spark.table(
    "workspace.silver.absenteeism"
)

print(
    f"Registros utilizados como origem da Gold: "
    f"{df_silver_gold.count()}"
)

Registros utilizados como origem da Gold: 740


In [0]:
%python
# --------------------------------------------
# 6.1 ABSENTEÍSMO POR MÊS
# --------------------------------------------

df_gold_month = (
    df_silver_gold
    .groupBy(
        "month_code",
        "month_name"
    )
    .agg(
        F.count("*").alias("absence_occurrences"),
        F.sum("absenteeism_time_in_hours").alias("total_absence_hours"),
        F.round(
            F.avg("absenteeism_time_in_hours"), 2
        ).alias("average_absence_hours")
    )
    .orderBy("month_code")
)

display(df_gold_month)

month_code,month_name,absence_occurrences,total_absence_hours,average_absence_hours
0,Não informado,3,0,0.0
1,Janeiro,50,222,4.44
2,Fevereiro,72,294,4.08
3,Março,87,765,8.79
4,Abril,53,482,9.09
5,Maio,64,400,6.25
6,Junho,54,411,7.61
7,Julho,67,734,10.96
8,Agosto,54,288,5.33
9,Setembro,53,292,5.51


In [0]:
%python
# --------------------------------------------
# 6.2 ABSENTEÍSMO POR DIA DA SEMANA
# --------------------------------------------

df_gold_weekday = (
    df_silver_gold
    .groupBy(
        "day_of_week_code",
        "day_of_week_name"
    )
    .agg(
        F.count("*").alias("absence_occurrences"),
        F.sum("absenteeism_time_in_hours").alias("total_absence_hours"),
        F.round(
            F.avg("absenteeism_time_in_hours"), 2
        ).alias("average_absence_hours")
    )
    .orderBy("day_of_week_code")
)

display(df_gold_weekday)

day_of_week_code,day_of_week_name,absence_occurrences,total_absence_hours,average_absence_hours
2,Segunda-feira,161,1489,9.25
3,Terça-feira,154,1229,7.98
4,Quarta-feira,156,1115,7.15
5,Quinta-feira,125,553,4.42
6,Sexta-feira,144,738,5.13


In [0]:
%python
# --------------------------------------------
# 6.3 ABSENTEÍSMO POR MOTIVO
# --------------------------------------------

df_gold_reason = (
    df_silver_gold
    .groupBy(
        "reason_code",
        "reason_description"
    )
    .agg(
        F.count("*").alias("absence_occurrences"),
        F.sum("absenteeism_time_in_hours").alias("total_absence_hours"),
        F.round(
            F.avg("absenteeism_time_in_hours"), 2
        ).alias("average_absence_hours")
    )
    .orderBy(
        F.col("total_absence_hours").desc()
    )
)

display(df_gold_reason)

reason_code,reason_description,absence_occurrences,total_absence_hours,average_absence_hours
13,Doenças do sistema musculoesquelético e tecido conjuntivo,55,842,15.31
19,"Lesões, envenenamentos e consequências de causas externas",40,729,18.23
23,Consulta médica,149,424,2.85
28,Consulta odontológica,112,335,2.99
11,Doenças do sistema digestivo,26,297,11.42
22,Acompanhamento do paciente,38,293,7.71
10,Doenças do sistema respiratório,25,276,11.04
26,Ausência injustificada,33,240,7.27
18,"Sintomas, sinais e achados clínicos e laboratoriais",21,217,10.33
12,Doenças da pele e tecido subcutâneo,8,187,23.38


In [0]:
%python
# --------------------------------------------
# 6.4 ABSENTEÍSMO POR COLABORADOR
# --------------------------------------------

# Total de horas da base
total_horas = (
    df_silver_gold
    .agg(F.sum("absenteeism_time_in_hours"))
    .first()[0]
)

df_gold_employee = (
    df_silver_gold
    .groupBy("id")
    .agg(
        F.count("*").alias("absence_occurrences"),
        F.sum("absenteeism_time_in_hours").alias("total_absence_hours"),
        F.round(
            F.avg("absenteeism_time_in_hours"), 2
        ).alias("average_absence_hours")
    )
    .withColumn(
        "percentage_total_hours",
        F.round(
            (F.col("total_absence_hours") / F.lit(total_horas)) * 100,
            2
        )
    )
    .orderBy(
        F.col("total_absence_hours").desc()
    )
)

display(df_gold_employee)

id,absence_occurrences,total_absence_hours,average_absence_hours,percentage_total_hours
3,113,482,4.27,9.41
14,29,476,16.41,9.29
11,40,450,11.25,8.78
28,76,347,4.57,6.77
34,55,344,6.25,6.71
36,34,311,9.15,6.07
20,42,306,7.29,5.97
9,8,262,32.75,5.11
24,30,254,8.47,4.96
22,46,253,5.5,4.94


In [0]:
%python
# --------------------------------------------
# 6.5 PERSISTÊNCIA DA CAMADA GOLD
# --------------------------------------------

tabelas_gold = {
    "agg_absenteeism_month": df_gold_month,
    "agg_absenteeism_weekday": df_gold_weekday,
    "agg_absenteeism_reason": df_gold_reason,
    "agg_absenteeism_employee": df_gold_employee
}

for nome_tabela, dataframe in tabelas_gold.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.gold.{nome_tabela}")
    )

    print(f"workspace.gold.{nome_tabela} gravada com sucesso.")

workspace.gold.agg_absenteeism_month gravada com sucesso.
workspace.gold.agg_absenteeism_weekday gravada com sucesso.
workspace.gold.agg_absenteeism_reason gravada com sucesso.
workspace.gold.agg_absenteeism_employee gravada com sucesso.


In [0]:
%python
# --------------------------------------------
# 6.6 VALIDAÇÃO DA CAMADA GOLD
# --------------------------------------------

tabelas_gold_validacao = [
    "agg_absenteeism_month",
    "agg_absenteeism_weekday",
    "agg_absenteeism_reason",
    "agg_absenteeism_employee"
]

for tabela in tabelas_gold_validacao:
    df_validacao = spark.table(f"workspace.gold.{tabela}")

    print(
        f"workspace.gold.{tabela}: "
        f"{df_validacao.count()} registros"
    )

workspace.gold.agg_absenteeism_month: 13 registros
workspace.gold.agg_absenteeism_weekday: 5 registros
workspace.gold.agg_absenteeism_reason: 28 registros
workspace.gold.agg_absenteeism_employee: 36 registros


In [0]:
%python
# Verificação dos códigos de motivo presentes nos dados
# O código 20, previsto na documentação da fonte, não possui registros no dataset.

display(
    spark.table("workspace.gold.agg_absenteeism_reason")
    .select(
        "reason_code",
        "reason_description"
    )
    .orderBy("reason_code")
)

reason_code,reason_description
0,Não especificado na documentação
1,Doenças infecciosas e parasitárias
2,Neoplasias
3,Doenças do sangue e transtornos imunológicos
4,"Doenças endócrinas, nutricionais e metabólicas"
5,Transtornos mentais e comportamentais
6,Doenças do sistema nervoso
7,Doenças dos olhos e anexos
8,Doenças do ouvido e processo mastoide
9,Doenças do sistema circulatório
